# Lecture 2 — Physical Regimes with Conditionals (Physics Logic in Code)

**Goal:**  
Use `if / elif / else` to encode **physics regime decisions**:
- bound vs unbound orbits
- escape vs capture
- stellar fusion thresholds
- sanity checks for valid inputs

Physics often looks like:
> "If condition A holds → use model A, otherwise use model B."


## Recap from Lecture 1

- Variables represent **physical quantities**
- Code executes mathematics
- Real numbers in computers are **floating-point approximations**

Today we add:
- **Boolean logic**: True / False
- **Conditionals**: `if / elif / else`
- **Safe comparisons** (avoid `==` for floats)


In [1]:
import math

G = 6.674e-11  # m^3 kg^-1 s^-2
M_sun = 1.989e30  # kg
R_sun = 6.9634e8  # m


## Booleans & Comparisons

Comparisons produce booleans:
- `>` `<` `>=` `<=`
- `==` `!=`  (⚠️ careful for floats)

You will use these to encode regime changes in physics.


In [3]:
T = 5800.1  # K
T > 5000, T < 3000, T == 5800.1


(True, False, True)

## Floating-point: Don't use `==`

Because floats are approximations, exact equality can fail.
Instead, compare with a tolerance:

\[
|a - b| < \varepsilon
\]


In [4]:
def approx_equal(a: float, b: float, tol: float = 1e-12) -> bool:
    return abs(a - b) < tol

approx_equal(0.1 + 0.2, 0.3), (0.1 + 0.2 == 0.3)


(True, False)

## Physics Regime Example 1: Escape vs Not Escape

Escape speed from a mass $\(M\) at distance \(r\):

$$
v_{\mathrm{esc}} = \sqrt{\frac{2GM}{r}}
$$

If:
- \(v \ge v_{\mathrm{esc}}\) → escape (unbound)
- \(v < v_{\mathrm{esc}}\) → bound (will not escape)

This is a clean `if/else` physics decision.


In [7]:
def escape_speed(M: float, r: float) -> float:
    v = math.sqrt(2 * G * M / r)
    return v

def escape_or_bound(M: float, r: float, v: float) -> str:
    vesc = escape_speed(M, r)
    if v >= vesc:
        return f"ESCAPE: v={v:.2e} m/s >= v_esc={vesc:.2e} m/s"
    else:
        return f"BOUND:  v={v:.2e} m/s <  v_esc={vesc:.2e} m/s"

# Example: near the Sun at 1 AU
r = 1.496e11
v = 30000  # m/s ~ 30 km/s (Earth orbital speed scale)
print(escape_or_bound(M_sun, r, v))
print(v)


BOUND:  v=3.00e+04 m/s <  v_esc=4.21e+04 m/s
30000


## Physics Regime Example 2: Orbit Type via Specific Orbital Energy

Specific orbital energy (per unit mass) in Newtonian gravity:

$$
\varepsilon = \frac{v^2}{2} - \frac{GM}{r}
$$

Interpretation:
- \(\varepsilon < 0\) → **bound** (elliptical or circular)
- \(\varepsilon = 0\) → **parabolic** (escape at exactly threshold)
- \(\varepsilon > 0\) → **hyperbolic** (unbound flyby)

This gives a physics-based `if/elif/else`.


In [8]:
def specific_orbital_energy(M: float, r: float, v: float) -> float:
    return 0.5 * v**2 - (G * M / r)

def classify_orbit(M: float, r: float, v: float, tol: float = 1e-10) -> str:
    eps = specific_orbital_energy(M, r, v)
    if eps < -tol:
        kind = "BOUND (elliptic/circular)"
    elif abs(eps) <= tol:
        kind = "PARABOLIC (threshold escape)"
    else:
        kind = "UNBOUND (hyperbolic)"
    return f"epsilon={eps:.3e} J/kg -> {kind}"

# Try three speeds at the same radius
r = 1.496e11
for v in [20000, 42000, 60000]:  # m/s
    print(v, classify_orbit(M_sun, r, v))


20000 epsilon=-6.873e+08 J/kg -> BOUND (elliptic/circular)
42000 epsilon=-5.339e+06 J/kg -> BOUND (elliptic/circular)
60000 epsilon=9.127e+08 J/kg -> UNBOUND (hyperbolic)


## Compound Conditions: Validity Checks

Physics inputs must be valid:
- distances \(r > 0\)
- masses \(M > 0\)
- speeds \(v \ge 0\)

Compound checks use:
- `and`, `or`, `not`

Good physics code fails early if inputs are nonsense.


In [ ]:
def safe_escape_speed(M: float, r: float) :
    if (M <= 0) or (r <= 0):
        raise ValueError("Physical inputs must satisfy M > 0 and r > 0.")
    return escape_speed(M, r)

# Demo: uncomment to see the error
v =safe_escape_speed(-1, 10)
v_new = v + 100.0
v_new


100.0

## Physics Regime Example 3: Fusion Thresholds (Mass-based)

A simple astrophysical regime classification by mass:

- \(M < 0.08 M_{\odot}\) : Brown dwarf (no sustained H fusion)
- \(0.08 \le M \lesssim 8 M_{\odot}\) : Typical stellar evolution (ends as white dwarf)
- \(M \gtrsim 8 M_{\odot}\) : Massive star (core-collapse supernova likely)

This is *not the full story* (metallicity, rotation, binarity matter),
but it's a good physics-flavoured conditional example.


In [ ]:
def star_mass_regime(M_in_Msun: float) -> str:
    if M_in_Msun < 0:
        return "Invalid: negative mass"
    if M_in_Msun < 0.08:
        return "Brown dwarf regime (no sustained H fusion)"
    elif M_in_Msun < 8.0:
        return "Low/intermediate-mass star regime (typical stellar evolution)"
    else:
        return "Massive star regime (core-collapse likely)"

for m in [0.05, 0.5, 1.0, 12.0]:
    print(m, "Msun ->", star_mass_regime(m))


## (Optional) `try/except`: Friendly Failure

When reading inputs from humans or files, conversions can fail.
Instead of crashing, we can catch the error and respond cleanly.

(You'll see this a lot when reading data files later.)


In [ ]:
def parse_float(s: str):
    try:
        
        return float(s)
    except ValueError:
        return None

tests = ["3.14", "ten", "1e-3"]
for t in tests:
    print(t, "->", parse_float(t))

def division(a: float, b: float) -> float:
    try:
        return a / b
    except ZeroDivisionError: 
        return float('inf')  # or some other sentinel value

print(division(10, 2))
print(division(10, 0))


3.14 -> 3.14
ten -> None
1e-3 -> 0.001


# ✍️ Exercises (Do these in-class)

## Exercise A — Escape vs Bound
Pick a central mass \(M\) and distance \(r\).
1) Compute \(v_{esc}\)
2) Test 3 different speeds
3) Print whether each case escapes

*Suggestion:* Use Earth mass and Earth radius (escape from Earth surface).

---

## Exercise B — Orbit Classification
At 1 AU around the Sun:
1) Find a speed that gives **bound**
2) Find a speed that gives **unbound**
3) Try to get **near-parabolic** (epsilon ≈ 0) using tolerance

---

## Exercise C — Regime Thinking
Write your own `if/elif/else` classifier for one of:
- Temperature regime (e.g., “cold / warm / hot plasma”)
- Density regime (e.g., “ISM / molecular cloud / stellar interior”)
Just pick thresholds and explain them in comments.
